In [1]:
import pandas as pd
bridges = pd.read_excel("data/BMMS_overview.xlsx")
roads = pd.read_csv("data/_roads3.csv")

In [2]:
roads_N1 = roads[roads["road"] == "N1"].copy()
bridges_N1 = bridges[bridges["road"] == "N1"].copy()

In [3]:
roads_N1 = roads_N1.sort_values("chainage").reset_index(drop=True)

roads_N1["start_km"] = roads_N1["chainage"]
roads_N1["end_km"]   = roads_N1["chainage"].shift(-1)

roads_N1 = roads_N1[roads_N1["road"] == roads_N1["road"].shift(-1)]

roads_N1["model_type"] = "link"

roads_N1["length"] = roads_N1["end_km"] - roads_N1["start_km"]

roads_N1 = roads_N1[[
    "road", "chainage", "model_type", "name", "lat", "lon",
    "length", "start_km", "end_km"]]

In [4]:
bridges_N1["length_km"] = bridges_N1["length"] / 1000
bridges_N1["start_km"] = bridges_N1["chainage"] - bridges_N1["length_km"] / 2
bridges_N1["end_km"]   = bridges_N1["chainage"] + bridges_N1["length_km"] / 2
bridges_N1["model_type"] = "bridge"
bridges_N1["length"] = bridges_N1["length_km"]

bridges_N1 = bridges_N1.rename(columns={"LRPName": "lrp"})

bridges_N1 = bridges_N1[[
    "road", "model_type", "name", "lat", "lon",
    "length", "condition", "start_km", "end_km"]]

In [5]:
cut_points = sorted(
    set(bridges_N1["start_km"].tolist() + bridges_N1["end_km"].tolist()))

split_links = []

bridge_intervals = [(row.start_km, row.end_km) for _, row in bridges_N1.iterrows()]

def inside_bridge(s, e):
    for bs, be in bridge_intervals:
        if s >= bs and e <= be:
            return True
    return False

for _, row in roads_N1.iterrows():
    s, e = row["start_km"], row["end_km"]

    internal_cuts = [x for x in cut_points if s < x < e]

    boundaries = [s] + internal_cuts + [e]

    for i in range(len(boundaries) - 1):
        seg_start = boundaries[i]
        seg_end   = boundaries[i+1]

        if inside_bridge(seg_start, seg_end):
            continue

        new_row = row.copy()
        new_row["start_km"] = seg_start
        new_row["end_km"]   = seg_end
        new_row["length"]   = seg_end - seg_start

        split_links.append(new_row)

In [6]:
roads_N1 = pd.DataFrame(split_links)

combined = pd.concat([roads_N1, bridges_N1], ignore_index=True)

combined = combined.sort_values("start_km").reset_index(drop=True)

full_N1 = combined[[
    "road", "model_type", "name",
    "lat", "lon", "length", "condition",
    "start_km", "end_km"]]

full_N1.loc[0, "model_type"] = full_N1.loc[0, "model_type"].replace("link", "source")
full_N1.loc[full_N1.shape[0]-1,"model_type"] = full_N1.loc[full_N1.shape[0]-1, "model_type"].replace("link", "sink")

full_N1["id"] = range(1_000_000, 1_000_000 + len(full_N1))

full_N1["length"] = full_N1["length"] * 1000


In [7]:
full_N1.head(30)

,road,model_type,name,lat,lon,length,condition,start_km,end_km,id
0,N1,source,Start of Road after Jatrabari Flyover infront...,23.706028,90.443333,814.00,NaN,0.000000,0.814000,1000000
1,N1,link,Box Culvert,23.702917,90.450417,8.00,NaN,0.814000,0.822000,1000001
2,N1,link,Intersection with Z1101,23.702778,90.450472,178.00,NaN,0.822000,1.000000,1000002
3,N1,link,Km post missing,23.702139,90.451972,794.35,NaN,1.000000,1.794350,1000003
4,N1,bridge,.,23.698739,90.458861,11.30,A,1.794350,1.805650,1000004
5,N1,link,Km post missing,23.702139,90.451972,194.35,NaN,1.805650,2.000000,1000005
6,N1,link,Km post missing,23.697889,90.460583,130.00,NaN,2.000000,2.130000,1000006
7,N1,link,Box culvert,23.697361,90.461667,870.00,NaN,2.130000,3.000000,1000007
8,N1,link,Km post missing,23.693833,90.469138,1000.00,NaN,3.000000,4.000000,1000008
9,N1,link,Km post missing,23.693611,90.478777,175.00,NaN,4.000000,4.175000,1000009


In [8]:
full_N1.to_csv("data/N1_AS2.csv", index=False)